# 68. c04에서 캐치 후 일어서는 동안의 원판과 체간
같은 30장을 따라가며 원판이 가까운 어깨보다 어디에 있는지, 몸통은 수직에 얼마나 가까워지는지 살펴봅니다. 캐치와 피니시는 트레이너의 1차 라벨입니다. 선수는 화면 왼쪽을 향하고 왼쪽 관절이 카메라에 가깝습니다. 원판은 베이스라인, 자세는 원본 YOLO26n-pose입니다.

먼저 프로젝트 루트에서 `uv run --frozen --env-file .env python -m wlcv.analyze c04_sn_f_01 --device mps`를 실행해 영상과 프레임별 CSV를 준비하세요. 아래에서는 기존 예측을 그대로 사용합니다.

In [ ]:
import json
from pathlib import Path

from IPython.display import Video, display
from wlcv.config import ROOT
from wlcv.data.frames import read_rgb
from wlcv.evaluation.catch_to_finish import VIDEO, landmarks, plot_interval, read_interval, write_clip
import matplotlib.pyplot as plt

folder = ROOT / "outputs/video_analysis" / VIDEO
# 최초 판독과 같은 실행을 우선한다. 다른 Mac에서는 가장 최근의 같은 조건 실행을 찾는다.
original_run = folder / "20261126-110143-542820"
for run in [original_run, *sorted(folder.glob("*/"), reverse=True)]:
    if not all((run / name).is_file() for name in ("summary.json", "frames.csv", "overlay.mp4")):
        continue
    settings = json.loads((run / "summary.json").read_text())
    if (settings.get("plate_mode"), settings.get("device"), settings.get("read_mode"),
        settings.get("plate_imgsz"), settings.get("pose_imgsz")) == ("single", "mps", "single-pass", 640, 640):
        break
else:
    raise FileNotFoundError("위의 Mac 분석 명령으로 단일 모델의 실행 결과를 준비하세요.")

out = ROOT / "outputs/catch_to_finish" / VIDEO / run.name
out.mkdir(parents=True, exist_ok=True)
rows = read_interval(run)
print("분석 실행 폴더", run)
print("원본 프레임 범위와 장수", rows.index.min(), rows.index.max(), len(rows))
print("거리·각도가 계산된 장수", rows[["forward_px", "trunk_angle_deg"]].count().to_dict())

### 어깨를 기준으로 한 앞뒤 거리
원판의 검출 박스 중점과 가까운 어깨의 가로 좌표 차이를 사용합니다. 양수는 선수 앞, 음수는 선수 뒤이며 단위는 원본 픽셀입니다. 체간 기울기는 가까운 골반에서 어깨를 이은 선과 화면 수직선이 이루는 각도입니다. 수직이면 0°이고 앞·뒤 기울기의 방향은 구분하지 않습니다.

78번은 견봉과 골반 소켓 모두 가림·해부학적 위치를 감안해 사람이 추정했습니다. 수동 추정점은 기준값 계산에 넣지 않습니다. 다른 프레임도 직접 판독으로 확인된 값과 구분해 읽으세요.

In [ ]:
rows.to_csv(out / "frames.csv")
key = landmarks(rows)
display(rows.loc[key, ["elapsed_s", "plate_cx_px", "shoulder_x_px", "forward_px",
                       "trunk_angle_deg", "shoulder_review", "hip_review"]].round(2))
change = rows[["forward_px", "trunk_angle_deg"]].iloc[-1] - rows[["forward_px", "trunk_angle_deg"]].iloc[0]
print("피니시 − 캐치", change.round(2).to_dict())

In [ ]:
fig = plot_interval(rows)
fig.savefig(out / "curves.png", dpi=165)
plt.show()

### 오버레이로 위치 확인
노란 가로선은 원판 중심과 어깨 사이의 가로 거리, 흰 점선은 어깨를 지나는 수직선, 하늘색 체간 선은 모델이 잡은 어깨와 골반입니다. 원본 한 장당 한 장을 담아 0.25배속으로 재생합니다. 아래 초 표시는 캐치 이후 원본 경과 시간입니다.

In [ ]:
clip = out / "overlay.mp4"
write_clip(run, rows, clip)
display(Video(str(clip), embed=True, height=590))

In [ ]:
fig, axes = plt.subplots(1, len(key), figsize=(3.7 * len(key), 6.7), layout="constrained", squeeze=False)
for ax, frame in zip(axes[0], key):
    ax.imshow(read_rgb(clip, frame - rows.index[0]))
    ax.set_title(f"Frame {frame}")
    ax.axis("off")
fig.savefig(out / "key_frames.jpg", dpi=130)
plt.show()

92번을 재생해 최대 간격이 원판과 어깨의 상대 위치에서 생긴 것을 확인하세요. 끝점의 차이만으로 중간의 간격을 설명하기 어렵습니다. 모델 출력만으로 동작의 정확도를 판정하거나 선수 간 기준을 만들 수는 없습니다.